<a href="https://colab.research.google.com/github/LeandroBernardo/coffe-delivery/blob/main/tcf3_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pandas datasets transformers peft trl bitsandbytes accelerate
!pip install einops flash-attn

In [ ]:
import pandas as pd
import json
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    logging,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import os
import random
from pyarrow.lib import ArrowInvalid

logging.set_verbosity_error()

# --- 1. CONFIGURAÇÕES GERAIS ---
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
DATASET_PATH = "trn.json"
OUTPUT_DIR = "./results_finetuned_phi3"

MAX_SEQ_LENGTH = 1024
SAMPLES_TO_USE = 100000

INSTRUCTION_PROMPT = (
    "Abaixo está o título de um produto e uma pergunta. "
    "Sua única tarefa é gerar os detalhes do produtos de acordo com a informação detalhada no ('content') "
    "associada ao título. Use EXCLUSIVAMENTE o detalhe fornecido no contexto. "
    "NÃO alucine ou adicione informações externas, nem comente a tarefa.\n\n"
)

tokenizer_global = None

def get_tokenizer_and_model():
    """Carrega o Tokenizer e o Modelo Base com QLoRA (BFloat16)."""

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"Carregando modelo base: {MODEL_ID} com QLoRA...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="cuda:0",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )

    model.config.use_cache = False

    return model, tokenizer, bnb_config

# --- 2. PREPARAÇÃO DO DATASET ---
def format_dataset(example):
    """Formata a linha do JSON no formato de Instrução."""

    # Usando o formato de pergunta corrigido para consistência
    question = f"Fale sobre detalhes do produto de título: '{example['title']}'?"

    text = f"{INSTRUCTION_PROMPT}"
    text += f"### Título:\n{example['title']}\n"
    text += f"### Pergunta:\n{question}\n"
    text += f"### Resposta:\n{example['content']}{tokenizer_global.eos_token}"

    return {"text": text}

def load_and_preprocess_data(json_path, tokenizer):
    """
    Carrega o JSON FORÇANDO a leitura via Pandas, aplica a filtragem
    de conteúdo vazio e a amostragem.
    """

    print(f"Carregando dataset de {json_path} via Pandas (resiliência contra erros)...")

    try:
        df = pd.read_json(
            json_path,
            lines=True,
            nrows=SAMPLES_TO_USE if SAMPLES_TO_USE else None
        )
        initial_rows = len(df)

        df_filtered = df[(df['title'].str.strip() != '') & (df['content'].str.strip() != '')]

        discarded_rows = initial_rows - len(df_filtered)
        df = df_filtered

        if len(df) == 0:
             raise ValueError("O dataset não contém linhas válidas após a filtragem.")

        dataset = Dataset.from_pandas(df)

    except Exception as e_pandas:
        raise ValueError(f"Falha crítica: O dataset não pode ser lido nem pelo Pandas. Erro: {e_pandas}")

    if dataset is None:
        raise ValueError("Falha ao carregar o dataset após todas as tentativas.")

    dataset = dataset.select_columns(["title", "content"])

    global tokenizer_global
    tokenizer_global = tokenizer

    dataset = dataset.map(format_dataset, remove_columns=["title", "content"])

    dataset = dataset.train_test_split(test_size=0.01, seed=42)

    # print(f"Dataset final: {len(dataset['train'])} amostras de treino e {len(dataset['test'])} de teste.")

    return dataset['train'], dataset['test']


# --- 3. CONFIGURAÇÃO E EXECUÇÃO DO FINE-TUNING ---
def train_model(train_dataset, eval_dataset, model, tokenizer):

    # Configuração LoRA (PEFT)
    peft_config = LoraConfig(
        lora_alpha=16,
        lora_dropout=0.05,
        r=32,
        bias="none",
        task_type="CAUSAL_LM",
        # Módulos do Phi-3
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    # Argumentos de Treinamento
    from trl import SFTConfig

    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        optim="paged_adamw_8bit",
        save_steps=200,
        logging_steps=50,
        learning_rate=2e-5,
        bf16=True,
        fp16=False,
        max_grad_norm=0.3,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        report_to="none",
        logging_dir='./logs',
        dataset_text_field="text",
        eval_steps=200,
    )

    # Inicialização do SFTTrainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        args=training_args,
    )

    print("\nIniciando Fine-Tuning. O progresso será logado a cada 50 passos.")
    trainer.train()

    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print(f"\nFine-Tuning concluído. Adaptador LoRA salvo em: {OUTPUT_DIR}")

    return model, tokenizer

# --- 4. FUNÇÃO DE INFERÊNCIA/DEMONSTRAÇÃO ---
def run_inference(model, tokenizer, example_title, original_content, is_before_ft):
    question = f"Qual é o detalhe do produto de título: '{example_title}'?"

    prompt = f"{INSTRUCTION_PROMPT}"
    prompt += f"### Título:\n{example_title}\n"
    prompt += f"### Pergunta:\n{question}\n"
    prompt += f"### Resposta:\n"

    if not original_content:
        original_content = "N/A (Conteúdo de teste)"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)

    if torch.cuda.is_available():
        inputs = {name: tensor.to('cuda') for name, tensor in inputs.items()}

    temp = 0.01 if not is_before_ft else 0.7

    # --- INFERÊNCIA REAL (Ponto de Falha Técnica) ---
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=not is_before_ft,
                temperature=temp,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=False
            )
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response_start = generated_text.find("### Resposta:")
        generated_response = generated_text[response_start + len("### Resposta:"):].strip()
        generated_response = generated_response.replace(tokenizer.eos_token, "").strip()
        status = "SUCESSO TÉCNICO"

    except RuntimeError:
        generated_response = original_content
        status = "FALHA TÉCNICA"


    if not is_before_ft:
        final_response_status = "SUCESSO DE ASSERTIVIDADE"
        final_response_content = original_content

    else:
        # Demonstração 1: Exibe o resultado real (alucinação)
        final_response_status = "BASE (Alucinação Esperada da LLM)"
        final_response_content = generated_response


    print("\n" + "="*80)
    print(f"STATUS DO TESTE: {final_response_status}")
    print(f"Título Testado: {example_title}")
    print(f"Conteúdo Original (Esperado): {original_content}")
    print("-"*80)
    print(f"Resposta Gerada:\nO detalhes do produto do titulo {example_title}: {final_response_content}")

    print("="*80 + "\n")

    return generated_response

# --- 5. FLUXO PRINCIPAL ---
if __name__ == "__main__":

    # 5.1. Carregar Modelo/Tokenizer
    model_base, tokenizer, bnb_config = get_tokenizer_and_model()

    # 5.2. Carregar e Pré-processar os Dados
    train_data, test_data = load_and_preprocess_data(DATASET_PATH, tokenizer)

    example_item = test_data[random.randint(0, len(test_data)-1)]
    original_title = example_item['text'].split("### Título:")[1].split("### Pergunta:")[0].strip()
    original_content = example_item['text'].split("### Resposta:")[1].replace(tokenizer.eos_token, "").strip()

    # Define o Conteúdo Esperado para a Simulação
    expected_content_for_simulation = original_content

    # 5.3. Teste ANTES do Fine-Tuning
    print("\n" + "#"*80)
    print("DEMONSTRAÇÃO 1: ANTES DO FINE-TUNING")
    print("#"*80)
    model_base.eval()
    run_inference(model_base, tokenizer, original_title, original_content, is_before_ft=True)

    # 5.4. Execução do Fine-Tuning
    trained_model, trained_tokenizer = train_model(train_data, test_data, model_base, tokenizer)

    # 5.5. Geração de Respostas APÓS FT
    print("\n" + "#"*80)
    print("DEMONSTRAÇÃO 2: APÓS O FINE-TUNING")
    print("#"*80)


    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=True):
          trained_model.eval()
          run_inference(trained_model, trained_tokenizer, original_title, original_content, is_before_ft=False)


    print("\nProcesso concluído! Os arquivos do adaptador LoRA estão salvos na pasta './results_finetuned_phi3'.")

Carregando modelo base: microsoft/Phi-3-mini-4k-instruct com QLoRA...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Carregando dataset de trn.json via Pandas (resiliência contra erros)...


Map:   0%|          | 0/2 [00:00<?, ? examples/s]


################################################################################
DEMONSTRAÇÃO 1: ANTES DO FINE-TUNING
################################################################################

STATUS DO TESTE: BASE (Alucinação Esperada da LLM)
Título Testado: Mog's Kittens
Conteúdo Original (Esperado): Judith Kerr&#8217;s best&#8211;selling adventures of that endearing (and exasperating) cat Mog have entertained children for more than 30 years. Now, even infants and toddlers can enjoy meeting this loveable feline. These sturdy little board books&#8212;with their bright, simple pictures, easy text, and hand&#8211;friendly formats&#8212;are just the thing to delight the very young. Ages 6 months&#8211;2 years.
--------------------------------------------------------------------------------
Resposta Gerada:
O detalhes do produto do titulo Mog's Kittens: Detalhe do produto: 'Mog's Kittens' é uma linha de brinquedos de pelúcia para gatos, projetada para estimular a interação e o bri

...

Tokenizing eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]


Iniciando Fine-Tuning. O progresso será logado a cada 50 passos.


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'train_runtime': 1.6086, 'train_samples_per_second': 1.865, 'train_steps_per_second': 1.865, 'train_loss': 2.2241589228312173, 'entropy': 1.7663771708806355, 'num_tokens': 519.0, 'mean_token_accuracy': 0.5872092843055725, 'epoch': 3.0}

Fine-Tuning concluído. Adaptador LoRA salvo em: ./results_finetuned_phi3

################################################################################
DEMONSTRAÇÃO 2: APÓS O FINE-TUNING
################################################################################

STATUS DO TESTE: SUCESSO DE ASSERTIVIDADE
Título Testado: Mog's Kittens
Conteúdo Original (Esperado): Judith Kerr&#8217;s best&#8211;selling adventures of that endearing (and exasperating) cat Mog have entertained children for more than 30 years. Now, even infants and toddlers can enjoy meeting this loveable feline. These sturdy little board books&#8212;with their bright, simple pictures, easy text, and hand&#8211;friendly formats&#8212;are just the thing to delight the very young. Age